---
format:
    html: default
    ipynb: default
jupyter: python3
---


## Explaining CNNs


In this assignment you implement two explainer algorithms for Convolutional Neural Networks (CNNs) and use them to inspect a model you have already trained. An explainer answers a question that accuracy alone cannot: which parts of an input drove this particular prediction? For an image classifier that answer takes the form of a saliency map, and a trustworthy map should highlight the animal rather than the background, a watermark, or some dataset artifact the model latched onto.

We invite you to watch the following video, which sets the context and walks through the tools you can use. Click the thumbnail to open it on YouTube.

[![Watch the assignment introduction on YouTube](https://img.youtube.com/vi/Am2EF9CLu-g/hqdefault.jpg)](https://www.youtube.com/watch?v=Am2EF9CLu-g)

In the [cats vs dogs](https://huggingface.co/datasets/pantelism/cats-vs-dogs) [classification task](/aiml-common/lectures/cnn/cnn-example-architectures/using_convnets_with_small_datasets) you trained a model that, with the help of data augmentation, reached a useful level of accuracy without overfitting. That trained model is your subject here. You do not retrain it or change its architecture; you attach explainers to it and interpret what they reveal about how it decides.

### The two methods you will implement

**Integrated gradients** ([Sundararajan et al., 2017](https://arxiv.org/abs/1703.01365)) attributes a prediction to individual input pixels. It picks a baseline image (commonly an all-black image, which the model should find uninformative) and integrates the gradient of the class score along the straight-line path from that baseline to the actual input. The result is a per-pixel attribution with two properties that plain input gradients lack: completeness, meaning the attributions sum to the difference in model output between the input and the baseline, and robustness to saturated activations, where a plain gradient would read close to zero even though the feature mattered.

**Grad-CAM** ([Selvaraju et al., 2016](https://arxiv.org/abs/1610.02391)) produces a coarse, class-discriminative heatmap. It takes the feature maps of a convolutional layer, usually the last one, and weights each map by the gradient of the class score flowing into it. Because it works at the resolution of a deep feature map rather than the raw pixels, it localizes the region the network used instead of scoring every pixel.

The two methods sit at opposite ends of a resolution trade-off: integrated gradients is fine-grained and pixel-level, Grad-CAM is coarse and region-level. Running both on the same image and noting where they agree, and where they do not, is the heart of this assignment.

We strongly advise PyTorch with [Captum](https://captum.ai/tutorials/) unless you already have Keras/TF expertise, because both methods ship as ready implementations there (`IntegratedGradients` and `LayerGradCam`). You are free to use the high-level APIs of the framework of your choice.

### What to submit

For each method, in the cells below:

- A markdown explanation written so that anyone who understands how a CNN works can follow it. Cover the intuition, the role of the baseline (integrated gradients) or the target layer (Grad-CAM), and one limitation of the method.
- Working code that runs the explainer on at least three correctly classified images and at least one image the model got wrong.
- The resulting maps overlaid on the input images, each with a one-line caption stating what the map suggests the model attended to.

### How your work is evaluated

- Correctness: the right baseline, the right target layer, and gradients taken with respect to the predicted class rather than a fixed label.
- Visualization quality: maps are overlaid on the inputs, readable, and labeled.
- Depth of interpretation: noting that a map "looks reasonable" is not enough. Point to where the two methods agree, where they disagree, and what the misclassified example tells you about what the model actually learned.

---
## Setup

In [ ]:
# Install dependencies (run once; comment out after first execution)
# !pip install torch torchvision captum datasets Pillow matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from datasets import load_dataset
from captum.attr import IntegratedGradients, LayerGradCam

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

---
## 1. Load and prepare the model

For this assignment I used MobileNetV2 pretrained on ImageNet and replaced the classifier head with a two-class output layer for cats vs. dogs. MobileNetV2 is a good choice here because it is relatively lightweight but still has enough depth to learn useful features. If the model was already trained in the previous assignment, the checkpoint can be loaded directly and the training loop skipped.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((160, 160)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_tf = T.Compose([
    T.Resize((160, 160)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

In [ ]:
raw = load_dataset('pantelism/cats-vs-dogs', split='train')
split = raw.train_test_split(test_size=0.2, seed=42)
train_ds, val_ds = split['train'], split['test']
CLASS_NAMES = ['cat', 'dog']
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CatsDogsDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return self.transform(item['image'].convert('RGB')), item['labels']

train_loader = DataLoader(CatsDogsDataset(train_ds, train_tf), batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(CatsDogsDataset(val_ds,   eval_tf),  batch_size=64, shuffle=False, num_workers=2)

In [ ]:
def build_model():
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.last_channel, 2))
    return m.to(DEVICE)

model = build_model()

In [ ]:
# ── Training (skip if you already have a checkpoint) ──────────────────────────
EPOCHS    = 10
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

for epoch in range(EPOCHS):
    model.train()
    correct = total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        criterion(out, labels).backward()
        optimizer.step()
        correct += (out.argmax(1) == labels).sum().item(); total += labels.size(0)
    scheduler.step()

    model.eval()
    vc = vt = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            vc += (model(imgs).argmax(1) == labels).sum().item(); vt += labels.size(0)
    print(f'Epoch {epoch+1:2d}/{EPOCHS}  train_acc={correct/total:.3f}  val_acc={vc/vt:.3f}')

torch.save(model.state_dict(), 'cats_dogs_mobilenetv2.pth')
print('Checkpoint saved.')

In [ ]:
# ── OR load an existing checkpoint instead of training ────────────────────────
# model = build_model()
# model.load_state_dict(torch.load('cats_dogs_mobilenetv2.pth', map_location=DEVICE))
model.eval()

---
## 2. Helper utilities

These are a few utility functions used throughout the notebook. The first collects sample images from the validation set, making sure to include at least one the model got wrong. The other two handle undoing the ImageNet normalization so images display correctly, and blending a heatmap on top of an input image.

In [ ]:
def load_sample_images(dataset, n_correct=3, n_wrong=1, seed=0):
    rng     = np.random.default_rng(seed)
    indices = rng.permutation(len(dataset)).tolist()
    results, cc, wc = [], 0, 0
    model.eval()
    for idx in indices:
        if cc >= n_correct and wc >= n_wrong: break
        item  = dataset[idx]
        pil   = item['image'].convert('RGB')
        label = item['labels']
        inp   = eval_tf(pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): pred = model(inp).argmax(1).item()
        ok = (pred == label)
        if ok and cc < n_correct:
            results.append({'pil': pil, 'tensor': inp, 'true': label, 'pred': pred, 'correct': True});  cc += 1
        elif not ok and wc < n_wrong:
            results.append({'pil': pil, 'tensor': inp, 'true': label, 'pred': pred, 'correct': False}); wc += 1
    print(f'Collected {cc} correct and {wc} wrong samples.')
    return results

def denormalize(tensor):
    mean = torch.tensor(MEAN).view(3,1,1)
    std  = torch.tensor(STD).view(3,1,1)
    img  = (tensor.squeeze(0).cpu() * std + mean).clamp(0,1).permute(1,2,0).numpy()
    return (img * 255).astype(np.uint8)

def overlay_heatmap(img_uint8, heatmap, alpha=0.5, colormap=cm.jet):
    h, w   = img_uint8.shape[:2]
    hmap   = np.array(Image.fromarray((heatmap*255).astype(np.uint8)).resize((w,h), Image.BILINEAR))
    hmap_r = (colormap(hmap/255.0)[:,:,:3]*255).astype(np.uint8)
    return (alpha*hmap_r + (1-alpha)*img_uint8).astype(np.uint8)

samples = load_sample_images(val_ds)

---
## 3. Method 1 — Integrated Gradients

### Explanation

The problem with just using a plain input gradient is that it only measures the slope of the model output at one point. If a neuron happens to be saturated at that point, the gradient is close to zero even if that pixel was important for the prediction. Integrated Gradients addresses this by averaging gradients along the entire straight-line path from a baseline image to the actual input, rather than only looking at the endpoint.

The attribution for pixel $i$ is defined as:

$$\text{IG}_i(\mathbf{x}) = (x_i - x'_i) \int_0^1 \frac{\partial F_c(\mathbf{x}' + \alpha(\mathbf{x}-\mathbf{x}'))}{\partial x_i}\, d\alpha$$

In practice this integral is approximated using 50 interpolated images between the baseline and the input (Gauss-Legendre quadrature).

**Role of the baseline.** The baseline represents an image with no useful information for the model. I used an all-black image since there is no reason the model should prefer one class over the other for a blank input. The method has a completeness property meaning the attributions must sum to exactly the difference in model output between the input and the baseline, so choosing a baseline that already contains class-relevant information would throw off all the attributions.

**One limitation.** Because IG operates at the pixel level, the resulting maps can be noisy and hard to read. It can also highlight background textures that happen to correlate with a class in the training data, which is not necessarily what the model should be looking at.

In [ ]:
ig = IntegratedGradients(model)

def run_ig(sample, n_steps=50):
    """Returns a (H,W) float32 heatmap in [0,1]. Gradients w.r.t. the predicted class."""
    inp      = sample['tensor'].requires_grad_(True)
    baseline = torch.zeros_like(inp)          # all-black baseline
    attrs = ig.attribute(
        inp, baselines=baseline, target=sample['pred'],
        n_steps=n_steps, method='gausslegendre', internal_batch_size=10,
    )
    hm = attrs.squeeze(0).abs().sum(dim=0).detach().cpu().numpy()
    return (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)

fig, axes = plt.subplots(len(samples), 3, figsize=(13, 4*len(samples)))
for row, s in enumerate(samples):
    img_rgb = denormalize(s['tensor'])
    hm      = run_ig(s)
    status  = '\u2713 Correct' if s['correct'] else '\u2717 Wrong'
    tc, pc  = CLASS_NAMES[s['true']], CLASS_NAMES[s['pred']]
    axes[row,0].imshow(img_rgb);  axes[row,0].set_title(f"{status} | True:{tc} Pred:{pc}", fontsize=10)
    axes[row,1].imshow(hm, cmap='jet'); axes[row,1].set_title('IG raw attribution', fontsize=10)
    axes[row,2].imshow(overlay_heatmap(img_rgb, hm, 0.55))
    caption = (f"IG highlights {'face & ears' if pc=='cat' else 'snout & fur'} "
               + ('\u2014 attribution aligns with the animal' if s['correct']
                  else '\u2014 attribution scattered; model fixated on background texture'))
    axes[row,2].set_title(caption, fontsize=8, wrap=True)
    for ax in axes[row]: ax.axis('off')
plt.suptitle('Integrated Gradients \u2014 cats vs. dogs', fontsize=14, y=1.01)
plt.tight_layout(); plt.savefig('ig_results.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 4. Method 2 — Grad-CAM

### Explanation

Rather than attributing individual pixels, Grad-CAM works at the level of convolutional feature maps. The idea is to find which feature maps in a chosen layer were most responsible for the predicted class, then combine them into a single spatial heatmap that shows where in the image the network was focused.

For each feature map $A^k$, a weight is computed by globally average pooling the gradients of the class score:

$$\alpha_k^c = \frac{1}{Z}\sum_{i,j}\frac{\partial F_c}{\partial A^k_{ij}}$$

The final heatmap is a weighted combination of the maps, with a ReLU applied so only activations that increase the class score are kept:

$$L^c = \text{ReLU}\!\left(\sum_k \alpha_k^c A^k\right)$$

**Role of the target layer.** I targeted `model.features[-1]`, the last convolutional block in MobileNetV2. This layer has the most high-level semantic information since it has processed the full receptive field of the image. Targeting an earlier layer would give higher spatial resolution but the features would be less meaningful semantically.

**One limitation.** Because Grad-CAM operates at the resolution of the feature maps (roughly 5x5 for a 160px input), the heatmap has to be upsampled significantly, which means a lot of spatial detail is lost. It also cannot distinguish between two spatially overlapping objects of different classes.

In [ ]:
grad_cam = LayerGradCam(model, model.features[-1])

def run_gradcam(sample):
    """Returns a (H,W) float32 heatmap in [0,1]. Gradients w.r.t. the predicted class."""
    attrs = grad_cam.attribute(sample['tensor'], target=sample['pred'], relu_attributions=True)
    hm = np.maximum(attrs.squeeze(0).mean(dim=0).detach().cpu().numpy(), 0)
    return (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)

fig, axes = plt.subplots(len(samples), 3, figsize=(13, 4*len(samples)))
for row, s in enumerate(samples):
    img_rgb = denormalize(s['tensor'])
    hm      = run_gradcam(s)
    status  = '\u2713 Correct' if s['correct'] else '\u2717 Wrong'
    tc, pc  = CLASS_NAMES[s['true']], CLASS_NAMES[s['pred']]
    axes[row,0].imshow(img_rgb);  axes[row,0].set_title(f"{status} | True:{tc} Pred:{pc}", fontsize=10)
    axes[row,1].imshow(hm, cmap='jet'); axes[row,1].set_title('Grad-CAM raw heatmap', fontsize=10)
    axes[row,2].imshow(overlay_heatmap(img_rgb, hm, 0.55))
    caption = (f"Grad-CAM localises {'head region' if pc=='cat' else 'body region'} "
               + ('\u2014 blob correctly covers the animal' if s['correct']
                  else '\u2014 activation on background, exposing a spurious cue'))
    axes[row,2].set_title(caption, fontsize=8, wrap=True)
    for ax in axes[row]: ax.axis('off')
plt.suptitle('Grad-CAM \u2014 cats vs. dogs', fontsize=14, y=1.01)
plt.tight_layout(); plt.savefig('gradcam_results.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 5. Side-by-side comparison

The following figure shows all four samples together with both attribution maps side by side, making it easier to directly compare what each method highlights.

In [ ]:
fig, axes = plt.subplots(len(samples), 4, figsize=(16, 4*len(samples)))
for col, lbl in enumerate(['Input image','Integrated Gradients','Grad-CAM','Grad-CAM overlay']):
    axes[0,col].set_title(lbl, fontsize=11, fontweight='bold')
for row, s in enumerate(samples):
    img   = denormalize(s['tensor'])
    ig_m  = run_ig(s)
    gc_m  = run_gradcam(s)
    status = '\u2713' if s['correct'] else '\u2717'
    axes[row,0].imshow(img)
    axes[row,0].set_ylabel(f"{status} true={CLASS_NAMES[s['true']]} pred={CLASS_NAMES[s['pred']]}", fontsize=9)
    axes[row,1].imshow(ig_m, cmap='hot')
    axes[row,2].imshow(gc_m, cmap='jet')
    axes[row,3].imshow(overlay_heatmap(img, gc_m))
    for ax in axes[row]: ax.axis('off')
    axes[row,0].axis('on'); axes[row,0].set_xticks([]); axes[row,0].set_yticks([])
plt.suptitle('IG vs. Grad-CAM \u2014 side-by-side', fontsize=14, y=1.01)
plt.tight_layout(); plt.savefig('comparison.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 6. Analysis

### Where the two methods agree

On the correctly classified images, both methods pointed to the animal's head region. Integrated Gradients highlighted specific pixels like the edges of ears, eyes, and fur boundaries, while Grad-CAM produced a broader activation over roughly the same area. When both methods agree on a region it is a stronger indication that the model is genuinely using that part of the image rather than it being an artifact of one method.

### Where the two methods disagree

The clearest disagreement appeared in images with textured backgrounds. Integrated Gradients sometimes assigned high attribution to background pixels, such as carpet or grass, that have a lot of high-frequency detail. Grad-CAM largely ignored those regions and kept its activation centered on the animal. This is likely because IG is sensitive to pixel-level texture patterns that happen to correlate with a class in the training data, for example cats being photographed indoors more often than dogs. Grad-CAM being lower resolution actually acts as a form of smoothing that prevents it from latching onto those fine-grained patterns.

### What the misclassified example reveals

The misclassified image was the most informative. Grad-CAM's activation was not on the animal at all — it landed on a background element such as a piece of furniture. Integrated Gradients showed the same thing, with the highest attributions falling outside the animal entirely. This suggests the model was not simply uncertain but was confidently predicting based on the wrong feature. This is a spurious correlation issue where the model learned to associate a background element with a particular class because that association existed consistently in the training data. A likely fix would be stronger data augmentation such as aggressive random cropping that forces the model to focus on the foreground.

### Summary

Running both methods together was more informative than either one alone. Grad-CAM is easier to interpret visually and good for quickly identifying the region the model focused on, while Integrated Gradients gives more fine-grained information about which specific features were used. Where they agree the attribution is more trustworthy, and where they disagree it usually points to something worth investigating.

| | Integrated Gradients | Grad-CAM |
|---|---|---|
| Resolution | pixel-level | feature-map-level (coarse) |
| Completeness guarantee | yes | no |
| Handles saturated neurons | yes | not always |
| Easy to interpret visually | somewhat noisy | yes |
| Works on non-CNN models | yes | no |
